[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/01_gpu_fundamentals/01.3_batch_and_instance_selection/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue?logo=)](https://molab.cloud/github/harshuljain13/llm-inference-at-scale/blob/master/content/01_gpu_fundamentals/01.3_batch_and_instance_selection/lab.ipynb)

# Lab 1.3: Batch Size and Instance Selection

This lab builds intuition for how batch size affects GPU memory, how to compare cost-per-token across instances, and how to select the right GPU given a model and latency SLA.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Model parameters (Llama 3.1 8B as reference)
MODEL_PARAMS_B = 8          # billion parameters
BYTES_PER_PARAM = 2         # FP16
WEIGHT_GB = MODEL_PARAMS_B * BYTES_PER_PARAM  # 16 GB

# KV cache parameters
NUM_LAYERS = 32
NUM_KV_HEADS = 8            # GQA
HEAD_DIM = 128
SEQ_LEN = 2048

def kv_cache_gb(batch_size, seq_len=SEQ_LEN):
    """KV cache memory in GB for a given batch size."""
    # 2 (K+V) * layers * kv_heads * head_dim * seq_len * batch * bytes_per_param
    bytes_total = 2 * NUM_LAYERS * NUM_KV_HEADS * HEAD_DIM * seq_len * batch_size * BYTES_PER_PARAM
    return bytes_total / (1024**3)

print(f"Weight memory: {WEIGHT_GB:.1f} GB")
print(f"KV cache per request (seq={SEQ_LEN}): {kv_cache_gb(1)*1024:.1f} MB")
print(f"KV cache at batch=32: {kv_cache_gb(32):.2f} GB")

## Exercise 1: VRAM Usage vs Batch Size

Weight memory is constant regardless of batch size. KV cache grows linearly. This plot shows where each GPU runs out of memory.

In [ ]:
batch_sizes = np.arange(1, 129)
weight_mem = np.full_like(batch_sizes, WEIGHT_GB, dtype=float)
kv_mem = np.array([kv_cache_gb(b) for b in batch_sizes])
total_mem = weight_mem + kv_mem

# GPU VRAM capacities
gpus = {"T4 (16 GB)": 16, "A10G (24 GB)": 24, "A100 (40 GB)": 40, "A100 (80 GB)": 80, "H100 (80 GB)": 80}

fig, ax = plt.subplots(figsize=(10, 6))
ax.fill_between(batch_sizes, 0, weight_mem, alpha=0.4, label="Weights (constant)", color="#2563eb")
ax.fill_between(batch_sizes, weight_mem, total_mem, alpha=0.4, label="KV Cache (linear)", color="#dc2626")
ax.plot(batch_sizes, total_mem, color="black", linewidth=2, label="Total VRAM")

for name, vram in gpus.items():
    ax.axhline(y=vram, linestyle="--", alpha=0.7, label=f"{name}")
    # Find max batch size for this GPU
    max_batch = np.searchsorted(total_mem, vram)
    if max_batch < len(batch_sizes):
        ax.annotate(f"max batch={max_batch}", xy=(max_batch, vram),
                    fontsize=8, ha="left", va="bottom")

ax.set_xlabel("Batch Size")
ax.set_ylabel("VRAM (GB)")
ax.set_title("VRAM Breakdown: Weights + KV Cache vs Batch Size\n(Llama 3.1 8B, FP16, seq=2048)")
ax.legend(loc="upper left", fontsize=9)
ax.set_xlim(1, 128)
ax.set_ylim(0, 90)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Exercise 2: Cost-per-Token Across GPU Instances

Throughput scales with batch size until memory or compute saturation. Cost-per-token = hourly_price / tokens_per_hour.

In [ ]:
# Instance specs: (name, vram_gb, hourly_cost_usd, tflops_fp16)
instances = [
    ("T4",        16,  0.53,   65),
    ("A10G",      24,  1.01,  125),
    ("L4",        24,  0.81,  121),
    ("A100-40",   40,  3.40,  312),
    ("A100-80",   80,  4.10,  312),
    ("H100",      80,  8.25,  990),
]

def max_batch_for_vram(vram_gb):
    """Max batch before OOM, reserving 10% for activations."""
    available = vram_gb * 0.9 - WEIGHT_GB
    if available <= 0:
        return 0
    per_request = kv_cache_gb(1)
    return int(available / per_request)

def tokens_per_hour(max_batch, tflops):
    """Approximate decode tokens/hour. Assumes memory-bound decode at ~30 tok/s/req baseline scaled by compute."""
    base_tok_per_s = 30  # single-request decode speed
    # Throughput scales linearly with batch until compute-bound
    compute_ratio = tflops / 65  # relative to T4
    effective_batch = min(max_batch, int(64 * compute_ratio))
    return effective_batch * base_tok_per_s * 3600

print(f"{'Instance':<10} {'VRAM':<8} {'Max Batch':<10} {'Tok/hr (M)':<12} {'$/hr':<8} {'$/M tokens':<10}")
print("-" * 60)
results = []
for name, vram, cost, tflops in instances:
    mb = max_batch_for_vram(vram)
    tph = tokens_per_hour(mb, tflops)
    cost_per_mtok = (cost / tph) * 1e6 if tph > 0 else float('inf')
    results.append((name, vram, mb, tph, cost, cost_per_mtok))
    print(f"{name:<10} {vram:<8} {mb:<10} {tph/1e6:<12.2f} {cost:<8.2f} ${cost_per_mtok:<9.3f}")

In [ ]:
# Visualize cost-per-million-tokens
names = [r[0] for r in results]
costs = [r[5] for r in results]
colors = ["#dc2626" if c == min(costs) else "#2563eb" for c in costs]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(names, costs, color=colors, edgecolor="black", linewidth=0.8)
ax.set_ylabel("Cost per Million Tokens ($)")
ax.set_title("Cost Efficiency: Llama 3.1 8B FP16 Decode\n(lower is better)")
ax.grid(True, axis="y", alpha=0.3)

for bar, cost in zip(bars, costs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"${cost:.3f}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

## Exercise 3: Instance Selection Decision Framework

Given a model size, sequence length, and latency SLA, systematically pick the cheapest GPU that meets all constraints.

In [ ]:
def select_instance(model_params_b, seq_len, target_batch, latency_sla_ms, precision_bytes=2):
    """
    Decision framework for GPU instance selection.
    
    Args:
        model_params_b: Model size in billions
        seq_len: Max sequence length
        target_batch: Required concurrent batch size
        latency_sla_ms: Max acceptable time-to-first-token (ms)
        precision_bytes: 2 for FP16, 1 for INT8
    """
    weight_gb = model_params_b * precision_bytes
    # KV cache scales with model dimensions (approximate from param count)
    layers = int(model_params_b * 4)  # rough heuristic
    kv_per_req_gb = (2 * layers * 8 * 128 * seq_len * precision_bytes) / (1024**3)
    total_vram_needed = weight_gb + (kv_per_req_gb * target_batch) + (weight_gb * 0.1)  # 10% activations
    
    # Prefill compute requirement (approximate)
    prefill_flops = 2 * model_params_b * 1e9 * seq_len  # forward pass FLOPs
    
    print(f"=== Instance Selection ===")
    print(f"Model: {model_params_b}B params, {precision_bytes}B precision")
    print(f"Sequence: {seq_len} tokens, Batch: {target_batch}")
    print(f"Latency SLA: {latency_sla_ms}ms TTFT")
    print(f"\nMemory requirement: {total_vram_needed:.1f} GB")
    print(f"Prefill compute: {prefill_flops/1e12:.1f} TFLOPs\n")
    
    print(f"{'Instance':<10} {'VRAM OK?':<10} {'Latency OK?':<12} {'Cost/hr':<10} {'Verdict'}")
    print("-" * 55)
    
    candidates = []
    for name, vram, cost, tflops in instances:
        vram_ok = vram >= total_vram_needed
        # TTFT ~ prefill_flops / (tflops * 1e12) * 1000 ms
        ttft_ms = (prefill_flops / (tflops * 1e12)) * 1000
        latency_ok = ttft_ms <= latency_sla_ms
        verdict = "✅ CANDIDATE" if (vram_ok and latency_ok) else "❌"
        if not vram_ok:
            verdict += " (OOM)"
        elif not latency_ok:
            verdict += f" (TTFT={ttft_ms:.0f}ms)"
        print(f"{name:<10} {'✓' if vram_ok else '✗':<10} {'✓' if latency_ok else '✗':<12} ${cost:<9.2f} {verdict}")
        if vram_ok and latency_ok:
            candidates.append((name, cost))
    
    if candidates:
        best = min(candidates, key=lambda x: x[1])
        print(f"\n→ Cheapest viable instance: {best[0]} at ${best[1]:.2f}/hr")
    else:
        print(f"\n→ No single GPU meets all constraints. Consider model parallelism or quantization.")
    return candidates

# Scenario: Llama 3.1 8B, 2048 seq, batch=16, 500ms TTFT SLA
_ = select_instance(model_params_b=8, seq_len=2048, target_batch=16, latency_sla_ms=500)

In [ ]:
# Try different scenarios
print("\n" + "="*60)
print("Scenario 2: 70B model, long context, strict latency")
print("="*60 + "\n")
_ = select_instance(model_params_b=70, seq_len=4096, target_batch=4, latency_sla_ms=200)

print("\n" + "="*60)
print("Scenario 3: 8B model quantized INT8, high throughput")
print("="*60 + "\n")
_ = select_instance(model_params_b=8, seq_len=2048, target_batch=64, latency_sla_ms=1000, precision_bytes=1)

## Key Takeaways

1. **Weights are fixed cost, KV cache is variable cost.** Batch size only affects KV memory.
2. **Bigger GPU ≠ cheaper inference.** Cost-per-token depends on how well you saturate the hardware.
3. **The selection framework is: memory filter → latency filter → cost sort.** Eliminate GPUs that OOM or miss SLA, then pick the cheapest survivor.
4. **Quantization shifts the entire curve** by halving weight memory, enabling larger batches on smaller GPUs.